# Drift / Schema Corruption Simulation — v3

Builds the reference dataset **in-memory**, the same way `05_automl.ipynb` does (via
`FeatureEngineeringClient` + `FeatureLookup` against `mlo.features.weather_daily_v3` and
`mlo.features.weather_labels`), instead of exporting to a CSV on DBFS and downloading it
locally. Serverless compute in this workspace doesn't have DBFS file read/write permissions
(`INSUFFICIENT_PRIVILEGES` on `SELECT` against files), so this sidesteps that entirely --
the dataframe never leaves the Spark session.

Reuses `get_feature_columns()`, `get_current_data()`, `run_drift_report()`, and
`print_summary()` directly from `drift_monitoring.py` (the v3 version, already written to
accept any reference dataframe + auto-detected feature columns) so the corruption/report
logic stays in one place -- this notebook only supplies the data differently.

**Note:** this only runs the data-loading half of `05_automl.ipynb` (the `FeatureLookup` /
`pdf = ts.load_df().toPandas()` part) -- it does NOT re-run model training. That keeps this
fast to re-run whenever you need a fresh reference sample.

## 0. Build the reference dataframe in-memory (same as 05_automl.ipynb, data-loading only)

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

fe = FeatureEngineeringClient()
V3 = "mlo.features.weather_daily_v3"

lookups = [FeatureLookup(table_name=V3, lookup_key=["station"], timestamp_lookup_key="date")]
ts = fe.create_training_set(
    df=spark.table("mlo.features.weather_labels"),
    feature_lookups=lookups,
    label="Bad",
)

reference_df = ts.load_df().toPandas().sort_values("date").reset_index(drop=True)
print(f"Loaded {len(reference_df)} rows, {len(reference_df.columns)} columns from {V3}")
reference_df.head()

## 1. Import the reusable pieces from drift_monitoring.py

In [0]:
import sys, os

try:
    from drift_monitoring import get_feature_columns, get_current_data, run_drift_report, print_summary, TARGET_COLUMN
except ImportError:
    # Fallback: add this notebook's repo folder to sys.path explicitly.
    repo_dir = os.path.dirname(os.path.abspath("__file__"))
    if repo_dir not in sys.path:
        sys.path.append(repo_dir)
    from drift_monitoring import get_feature_columns, get_current_data, run_drift_report, print_summary, TARGET_COLUMN

feature_columns = get_feature_columns(reference_df)
print(f"Auto-detected {len(feature_columns)} feature columns:")
print(feature_columns)

## 2. Corruption scenarios (v3-generic)

v3 has ~40 dynamic columns (multi-station, could shift as the nightly pipeline runs), so
these scenarios work by pattern-matching against whatever columns actually exist --
same approach as `get_current_data()`'s `perturb_matching()` -- rather than hardcoding
specific column names like the old v2 script did.

In [0]:
import numpy as np

RANDOM_STATE = 1


def corrupt_sensor_shift(df, feature_columns):
    """Reuses drift_monitoring.py's own drift injection -- the single
    baseline scenario it already implements."""
    corrupted = get_current_data(df, feature_columns, inject_drift=True)
    return corrupted, "Sensor/seasonal shift (via drift_monitoring.get_current_data)"


def corrupt_missing_data(df, feature_columns, frac=0.3, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    out = df.sample(n=min(200, len(df)), random_state=random_state).copy()
    for col in feature_columns:
        mask = rng.random(len(out)) < frac
        out.loc[mask, col] = np.nan
    return out, f"Missing data: ~{int(frac * 100)}% of values nulled per feature"


def corrupt_schema_change(df, feature_columns):
    """Renames one column, drops another -- picked dynamically since the
    v3 column list can shift as the nightly pipeline runs."""
    out = df.sample(n=min(200, len(df)), random_state=RANDOM_STATE).copy()
    if len(feature_columns) < 2:
        return out, "Schema change: skipped, not enough feature columns"
    rename_col, drop_col = feature_columns[0], feature_columns[-1]
    out = out.rename(columns={rename_col: f"{rename_col}_renamed"})
    out = out.drop(columns=[drop_col])
    return out, f"Schema change: {rename_col} renamed, {drop_col} dropped"


def corrupt_unit_error(df, feature_columns):
    out = df.sample(n=min(200, len(df)), random_state=RANDOM_STATE).copy()
    wind_cols = [c for c in feature_columns if "AWND" in c or "WSF" in c][:1]
    for c in wind_cols:
        out[c] = out[c] * 1.609
    label = f"Unit error: {wind_cols} reported in km/h instead of expected units" if wind_cols else \
        "Unit error: skipped, no wind-speed column matched"
    return out, label


def corrupt_stuck_sensor(df, feature_columns):
    out = df.sample(n=min(200, len(df)), random_state=RANDOM_STATE).copy()
    temp_cols = [c for c in feature_columns if "TMIN" in c][:1] or feature_columns[:1]
    for c in temp_cols:
        if len(out) > 0:
            out[c] = out[c].iloc[0]
    return out, f"Stuck sensor: {temp_cols} frozen at a single repeated value"


def corrupt_outliers(df, feature_columns, frac=0.05, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    out = df.sample(n=min(200, len(df)), random_state=random_state).copy()
    temp_cols = [c for c in feature_columns if "TMAX" in c][:1] or feature_columns[:1]
    n_outliers = max(1, int(len(out) * frac))
    idx = rng.choice(out.index, size=n_outliers, replace=False)
    for c in temp_cols:
        out.loc[idx, c] = out.loc[idx, c] * rng.uniform(3, 5, size=n_outliers)
    return out, f"Outliers: {n_outliers} rows with {temp_cols} inflated 3-5x"


SCENARIOS = {
    "sensor_shift": corrupt_sensor_shift,
    "missing_data": corrupt_missing_data,
    "schema_change": corrupt_schema_change,
    "unit_error": corrupt_unit_error,
    "stuck_sensor": corrupt_stuck_sensor,
    "outliers": corrupt_outliers,
}

## 3. Run every scenario and report drift

In [0]:
corrupted_results = {}

for name, fn in SCENARIOS.items():
    corrupted_df, description = fn(reference_df, feature_columns)
    corrupted_results[name] = corrupted_df
    print(f"\n[{name}] {description}")

    shared_cols = [c for c in feature_columns if c in corrupted_df.columns]
    result = run_drift_report(reference_df, corrupted_df, shared_cols)
    print_summary(result)

    # run_drift_report() always writes to the same DRIFT_REPORT_PATH -- rename
    # per scenario so each report survives the next loop iteration.
    import shutil
    shutil.move("drift_report.html", f"drift_report_{name}.html")
    print(f"  -> saved drift_report_{name}.html")

## 4. API payload export

In [0]:
def to_api_payload(df, feature_columns):
    cols = [c for c in feature_columns if c in df.columns]
    return df[cols].to_dict(orient="records")

for name, corrupted_df in corrupted_results.items():
    payload = to_api_payload(corrupted_df, feature_columns)
    print(f"{name}: {len(payload)} records ready -- payload[0] preview:")
    print(payload[0] if payload else "(empty)")